In [1]:
import pandas as pd
import re

#Chargement du fichier
df = pd.read_csv('BDD tp qualité de donnée.csv', sep=None, engine='python')

ETAPE 1 : ANALYSE DU DATASET

In [ ]:
#Affiche les 5 premieres lignes
df.head()

,id,nom,prenom,email,date_naissance,pays,code_postal,chiffre_affaires_total,nb_projets,dernier_contact
0,1,Nguyen,Paul,paul.nguyen@domain.com,%d/%m/%Y,Tunisie,16000,1 235,0,2024-01-10
1,2,Haddad,Lucas,lucas.haddad@yahoo.com,%Y-%m-%d,france,01300,1905 €,0,10/01/2024
2,3,Dupont,Awa,awa.dupont@gmail.com,%Y-%m-%d,TN,75 000,1144,17,10/01/2024
3,4,BENALI,sophie,sophie.benali@gmail.com,%Y-%m-%d,MA,31000,8 751,20,10/01/2024
4,5,Martin,Amine,amine.martin@domain.com,%Y-%m-%d,TN,75 000,4462,15,10/01/2024


In [3]:
#Affiche les types de donnees
df.dtypes

id                        int64
nom                         str
prenom                      str
email                       str
date_naissance              str
pays                        str
code_postal                 str
chiffre_affaires_total      str
nb_projets                int64
dernier_contact             str
dtype: object

In [4]:
#Affiche la taille (lignes, colonnes)
df.shape

(500, 10)

In [5]:
#Affiche resume general
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 10 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   id                      500 non-null    int64
 1   nom                     500 non-null    str  
 2   prenom                  500 non-null    str  
 3   email                   500 non-null    str  
 4   date_naissance          500 non-null    str  
 5   pays                    500 non-null    str  
 6   code_postal             500 non-null    str  
 7   chiffre_affaires_total  500 non-null    str  
 8   nb_projets              500 non-null    int64
 9   dernier_contact         500 non-null    str  
dtypes: int64(2), str(8)
memory usage: 39.2 KB


ETAPE 2 : GESTION DES VALEURS MANQUANTES

a- Calcul du nombre de valeurs manquantes

In [8]:
# Calcul du nmbre de valeurs manquantes par colonne
df.isnull().sum()

id                        0
nom                       0
prenom                    0
email                     0
date_naissance            0
pays                      0
code_postal               0
chiffre_affaires_total    0
nb_projets                0
dernier_contact           0
dtype: int64

Calcul du pourcentage

In [9]:
# Calcul du pourcentage
df.isnull().sum() / len(df) * 100

id                        0.0
nom                       0.0
prenom                    0.0
email                     0.0
date_naissance            0.0
pays                      0.0
code_postal               0.0
chiffre_affaires_total    0.0
nb_projets                0.0
dernier_contact           0.0
dtype: float64

In [10]:
df['date_naissance'].value_counts()

date_naissance
%Y-%m-%d    178
%d/%m/%Y    165
%d-%m-%Y    157
Name: count, dtype: int64

In [11]:
df['dernier_contact'].value_counts()
df['pays'].value_counts()

pays
MA         65
TN         64
FR         60
Algerie    57
france     55
DZ         55
France     54
Tunisie    45
Maroc      45
Name: count, dtype: int64

c- Identification des lignes critiques

In [12]:
# IDENTIFICATION DES LIGNES CRITIQUES
# On crée une colonne qui compte les problèmes par ligne
df['nb_problemes'] = 0

# Problème 1 : date_naissance contient un masque de format (pas une vraie date)
df['nb_problemes'] += df['date_naissance'].isin(['%Y-%m-%d', '%d/%m/%Y', '%d-%m-%Y']).astype(int)

# Problème 2 : pays non harmonisé (pas un code ISO à 2 lettres)
df['nb_problemes'] += (~df['pays'].isin(['FR', 'TN', 'MA', 'DZ'])).astype(int)

# Problème 3 : email invalide (contient un espace ou n'a pas de @)
df['nb_problemes'] += (~df['email'].str.match(r'^[^\s@]+@[^\s@]+\.[^\s@]+')).astype(int)

# Afficher les lignes avec le plus de problèmes
df.sort_values('nb_problemes', ascending=False).head(10)

,id,nom,prenom,email,date_naissance,pays,code_postal,chiffre_affaires_total,nb_projets,dernier_contact,nb_problemes
122,123,Martin,amine,amine.martingmail.com,%d-%m-%Y,Maroc,16000,8988 €,1,2024/01/10,3
430,431,Haddad,Jean,jean haddad@domain.com,%Y-%m-%d,france,16000,12 250,2,10/01/2024,3
42,43,MOREAU,Amine,amine moreau@domain.com,%Y-%m-%d,France,75000,11119,8,10/01/2024,3
154,155,Dupont,Sophie,sophie.dupontgmail.com,%d/%m/%Y,France,75000,13852,0,2024/01/10,3
478,479,Durand,Claire,claire durand@gmail.com,%d-%m-%Y,Tunisie,75 000,8960,20,10/01/2024,3
420,421,Martin,lina,lina martin@domain.com,%Y-%m-%d,Maroc,31000,5 011,7,10/01/2024,3
443,444,Leroy,sophie,sophie leroy@domain.com,%d-%m-%Y,France,75 000,6560,2,2024/01/10,3
445,446,Petit,Amine,amine.petitgmail.com,%d-%m-%Y,Algerie,75000,13935,14,2024-01-10,3
6,7,Diallo,marie,marie diallo@outlook.com,%d/%m/%Y,Tunisie,75000,3962,19,2024-01-10,3
147,148,Leroy,Amine,amine leroy@gmail.com,%d/%m/%Y,Algerie,75000,6354,20,2024/01/10,3


ETAPE 3 : GESTION DES DOUBLONS

a- Detection des doublons

In [13]:
# Doublons complets (lignes 100% identiques)
print("Doublons complets :", df.duplicated().sum())

# Doublons métier sur l'email
print("Doublons sur email :", df.duplicated(subset='email').sum())

Doublons complets : 0
Doublons sur email : 133


b- Proposition des strategies de traitement

In [14]:
#Strategie de traitement
# Voir un exemple de doublon métier
email_duplique = df[df.duplicated(subset='email', keep=False)]
email_duplique.sort_values('email').head(6)

,id,nom,prenom,email,date_naissance,pays,code_postal,chiffre_affaires_total,nb_projets,dernier_contact,nb_problemes
147,148,Leroy,Amine,amine leroy@gmail.com,%d/%m/%Y,Algerie,75000,6354,20,2024/01/10,3
25,26,LEROY,Amine,amine leroy@gmail.com,%d/%m/%Y,France,75000,4047,19,2024/01/10,3
193,194,Martin,Amine,amine martin@yahoo.com,%d/%m/%Y,france,31000,2983,5,2024-01-10,3
245,246,MARTIN,amine,amine martin@yahoo.com,%Y-%m-%d,Algerie,01300,13 106,11,2024-01-10,3
451,452,Benali,Amine,amine.benali@yahoo.com,%Y-%m-%d,MA,31000,14922 €,18,10/01/2024,1
297,298,Benali,amine,amine.benali@yahoo.com,%d/%m/%Y,TN,75000,6158 €,8,2024/01/10,1


ETAPE 4 : DETECTION DES ANOMALIES

a- Analyse des emails invalides

In [15]:
# Détecter les emails invalides
masque_email = ~df['email'].str.match(r'^[^\s@]+@[^\s@]+\.[^\s@]+')
print("Emails invalides :", masque_email.sum())

# Afficher les emails invalides
df[masque_email]['email'].head(10)

Emails invalides : 130


6     marie diallo@outlook.com
11       jean martin@yahoo.com
12    marie durand@outlook.com
13      lucas benali@yahoo.com
16       amine.moreaugmail.com
19     paul benali@outlook.com
22        paul.benaliyahoo.com
23      marie diallo@gmail.com
24      sophie petit@yahoo.com
25       amine leroy@gmail.com
Name: email, dtype: str

b- Analyse des dates incoherentes

In [16]:
# date_naissance
print("Valeurs date_naissance :")
print(df['date_naissance'].value_counts())

# dernier_contact
print("\nFormats dernier_contact :")
print(df['dernier_contact'].value_counts())

Valeurs date_naissance :
date_naissance
%Y-%m-%d    178
%d/%m/%Y    165
%d-%m-%Y    157
Name: count, dtype: int64

Formats dernier_contact :
dernier_contact
2024/01/10    187
10/01/2024    158
2024-01-10    155
Name: count, dtype: int64


c- Analyse des valeurs numeriques incorrectes

In [17]:
# Voir les valeurs brutes du chiffre d'affaires
df['chiffre_affaires_total'].value_counts().head(15)

chiffre_affaires_total
13852     2
11657     2
10131     2
6587 €    2
1 235     1
1905 €    1
1144      1
8 751     1
4462      1
6396      1
3962      1
4562 €    1
7542      1
9 151     1
10719     1
Name: count, dtype: int64

d- Analyse incoherence metiers

In [18]:
# Un client avec 0 projets mais un CA positif, c'est logiquement impossible
incoherence = df[(df['nb_projets'] == 0) & (df['chiffre_affaires_total'] != '0')]
print("Incohérences métier :", len(incoherence))
incoherence.head(5)

Incohérences métier : 25


,id,nom,prenom,email,date_naissance,pays,code_postal,chiffre_affaires_total,nb_projets,dernier_contact,nb_problemes
0,1,Nguyen,Paul,paul.nguyen@domain.com,%d/%m/%Y,Tunisie,16000,1 235,0,2024-01-10,2
1,2,Haddad,Lucas,lucas.haddad@yahoo.com,%Y-%m-%d,france,01300,1905 €,0,10/01/2024,2
62,63,DUPONT,amine,amine.dupontoutlook.com,%d/%m/%Y,france,16000,1696 €,0,2024-01-10,3
81,82,Leroy,Paul,paul.leroydomain.com,%Y-%m-%d,France,01300,8788,0,2024-01-10,3
124,125,Leroy,Awa,awa.leroy@domain.com,%Y-%m-%d,TN,75 000,1317,0,2024/01/10,1


ETAPE 5 : HARMONISATION DES DONNEES

a- Correction minuscules/majuscules

In [19]:
# D'abord observer ce qu'on a
print(df['pays'].value_counts())

# Créer un dictionnaire de correspondance
pays_map = {
    'france' : 'FR',
    'France' : 'FR',
    'Tunisie': 'TN',
    'Maroc'  : 'MA',
    'Algerie': 'DZ'
}

# Appliquer le remplacement
df['pays'] = df['pays'].replace(pays_map)

# Vérifier le résultat
print(df['pays'].value_counts())

pays
MA         65
TN         64
FR         60
Algerie    57
france     55
DZ         55
France     54
Tunisie    45
Maroc      45
Name: count, dtype: int64
pays
FR    169
DZ    112
MA    110
TN    109
Name: count, dtype: int64


b - Harmonisation de la casse des noms/prenoms

In [20]:
# Avant
print(df['nom'].value_counts().head(5))
print(df['prenom'].value_counts().head(5))

# Corriger
df['nom']    = df['nom'].str.strip().str.title()
df['prenom'] = df['prenom'].str.strip().str.title()

# Après
print(df['nom'].value_counts().head(5))

nom
Diallo    54
Haddad    45
Martin    42
Dupont    40
Moreau    31
Name: count, dtype: int64
prenom
Sophie    49
Paul      42
Amine     42
Lina      31
Claire    31
Name: count, dtype: int64
nom
Diallo    64
Haddad    58
Dupont    58
Martin    56
Benali    49
Name: count, dtype: int64


c- Nettoyage des emails

In [21]:
# Supprimer les espaces et mettre en minuscules
df['email'] = df['email'].str.replace(' ', '').str.lower().str.strip()

# Vérifier combien d'emails sont encore invalides après nettoyage
masque = ~df['email'].str.match(r'^[^\s@]+@[^\s@]+\.[^\s@]+')
print("Emails encore invalides :", masque.sum())
df[masque]['email'].head(5)

Emails encore invalides : 43


16      amine.moreaugmail.com
22       paul.benaliyahoo.com
29     sophie.benaliyahoo.com
52    sophie.leroyoutlook.com
57     claire.martinyahoo.com
Name: email, dtype: str

d- Nettoyage du code postal

In [22]:
# Supprimer les espaces (75 000 → 75000)
df['code_postal'] = df['code_postal'].str.replace(' ', '')

# Vérifier
print(df['code_postal'].value_counts())

code_postal
75000    184
16000    120
31000    106
01300     90
Name: count, dtype: int64


e- Harmoniser dernier_contact

In [23]:
# Convertir en date (Pandas détecte les formats automatiquement)
df['dernier_contact'] = pd.to_datetime(df['dernier_contact'], dayfirst=True, errors='coerce')

# Vérifier
print(df['dernier_contact'].dtype)
print(df['dernier_contact'].value_counts())

datetime64[us]
dernier_contact
2024-10-01    155
Name: count, dtype: int64


ETAPE 6 : TRANSFORMATION DES DONNEES

a- Nettoyage et convertion du chiffre_affaires_total

In [24]:
# Avant - voir les valeurs brutes
print(df['chiffre_affaires_total'].head(10))
print("Type actuel :", df['chiffre_affaires_total'].dtype)

# Nettoyer : supprimer €, espaces, puis convertir en nombre
df['chiffre_affaires_total'] = (
    df['chiffre_affaires_total']
    .str.replace('€', '', regex=False)   # supprimer le symbole €
    .str.replace(' ', '', regex=False)   # supprimer les espaces
    .str.strip()                          # supprimer espaces en début/fin
    .astype(float)                        # convertir en nombre décimal
)

# Après
print(df['chiffre_affaires_total'].head(10))
print("Type après conversion :", df['chiffre_affaires_total'].dtype)

0     1 235
1    1905 €
2      1144
3     8 751
4      4462
5      6396
6      3962
7    4562 €
8      7542
9     9 151
Name: chiffre_affaires_total, dtype: str
Type actuel : str
0    1235.0
1    1905.0
2    1144.0
3    8751.0
4    4462.0
5    6396.0
6    3962.0
7    4562.0
8    7542.0
9    9151.0
Name: chiffre_affaires_total, dtype: float64
Type après conversion : float64


b - Traitement de la date_naissance

In [25]:
# Remplacer toutes les valeurs par NaT (valeur manquante de type date)
df['date_naissance'] = pd.NaT

# Vérifier
print(df['date_naissance'].dtype)
print(df['date_naissance'].isnull().sum(), "valeurs manquantes")

datetime64[ns]
500 valeurs manquantes


c- verification de tous les types finaux

In [26]:
# Résumé final des types
print(df.dtypes)
print()
print(df.head())

id                                 int64
nom                                  str
prenom                               str
email                                str
date_naissance            datetime64[ns]
pays                                 str
code_postal                          str
chiffre_affaires_total           float64
nb_projets                         int64
dernier_contact           datetime64[us]
nb_problemes                       int64
dtype: object

   id     nom  prenom                    email date_naissance pays  \
0   1  Nguyen    Paul   paul.nguyen@domain.com            NaT   TN   
1   2  Haddad   Lucas   lucas.haddad@yahoo.com            NaT   FR   
2   3  Dupont     Awa     awa.dupont@gmail.com            NaT   TN   
3   4  Benali  Sophie  sophie.benali@gmail.com            NaT   MA   
4   5  Martin   Amine  amine.martin@domain.com            NaT   TN   

  code_postal  chiffre_affaires_total  nb_projets dernier_contact  \
0       16000                  1235.0       

ETAPE 7 : SCORE DE QUALITE

Ici on va creer un indicateur global qui mesure la qualite du dataste , on va se baser sur 3 dimension pour le faire.

a- Completude

In [27]:
# Calcul de la complétude
completude = 1 - df.isnull().mean().mean()
print(f"Complétude : {completude * 100:.2f}%")

Complétude : 84.64%


b- Validite

In [28]:
# Email valide
email_valide = df['email'].str.match(r'^[^\s@]+@[^\s@]+\.[^\s@]+')

# Pays valide (code ISO)
pays_valide = df['pays'].isin(['FR', 'TN', 'MA', 'DZ'])

# Score de validité
validite = (email_valide & pays_valide).mean()
print(f"Validité : {validite * 100:.2f}%")

Validité : 91.40%


c- Coherence

In [29]:
# Incohérence : CA > 0 mais nb_projets = 0
coherence = ~((df['nb_projets'] == 0) & (df['chiffre_affaires_total'] > 0))
coherence_score = coherence.mean()
print(f"Cohérence : {coherence_score * 100:.2f}%")

Cohérence : 95.00%


d- Score global

In [30]:
# Moyenne des 3 dimensions
score_global = (completude + validite + coherence_score) / 3
print(f"\n{'='*40}")
print(f"  Complétude  : {completude * 100:.2f}%")
print(f"  Validité    : {validite * 100:.2f}%")
print(f"  Cohérence   : {coherence_score * 100:.2f}%")
print(f"{'='*40}")
print(f"  Score global : {score_global * 100:.2f}%")
print(f"{'='*40}")


  Complétude  : 84.64%
  Validité    : 91.40%
  Cohérence   : 95.00%
  Score global : 90.35%


ETAPE 8 : AUTOMATISATION

Ici j'ai decide de regrouper tout ce qu'on a fait dans une seule fonction reutilisable. Ainsi, si je recois un new file demain, on lance juste cette fonction.

a- Fonction d'analyse

In [31]:
def analyser_dataset(filepath):
    """
    Analyse automatiquement un dataset CSV
    et retourne un rapport de qualité
    """
    # Chargement
    df = pd.read_csv(filepath, sep=None, engine='python')
    
    rapport = {}
    
    # Infos générales
    rapport['nb_lignes']   = len(df)
    rapport['nb_colonnes'] = len(df.columns)
    
    # Valeurs manquantes
    rapport['valeurs_manquantes'] = df.isnull().sum().to_dict()
    
    # Doublons
    rapport['doublons_complets'] = df.duplicated().sum()
    rapport['doublons_email']    = df.duplicated(subset='email').sum()
    
    # Emails invalides
    rapport['emails_invalides'] = (
        ~df['email'].str.match(r'^[^\s@]+@[^\s@]+\.[^\s@]+')
    ).sum()
    
    # Pays non harmonisés
    rapport['pays_non_harmonises'] = (
        ~df['pays'].isin(['FR', 'TN', 'MA', 'DZ'])
    ).sum()
    
    return rapport

b - Fonction de nettoyage

In [32]:
def nettoyer_dataset(df):
    """
    Applique toutes les corrections sur le dataset
    """
    # Harmoniser les pays
    pays_map = {
        'france': 'FR', 'France': 'FR',
        'Tunisie': 'TN', 'Maroc': 'MA',
        'Algerie': 'DZ'
    }
    df['pays'] = df['pays'].replace(pays_map)
    
    # Harmoniser la casse
    df['nom']    = df['nom'].str.strip().str.title()
    df['prenom'] = df['prenom'].str.strip().str.title()
    
    # Nettoyer les emails
    df['email'] = df['email'].str.replace(' ', '').str.lower().str.strip()
    
    # Nettoyer le code postal
    df['code_postal'] = df['code_postal'].str.replace(' ', '')
    
    # Convertir dernier_contact en date
    df['dernier_contact'] = pd.to_datetime(
        df['dernier_contact'], dayfirst=True, errors='coerce'
    )
    
    # Nettoyer chiffre_affaires_total
    df['chiffre_affaires_total'] = (
        df['chiffre_affaires_total']
        .str.replace('€', '', regex=False)
        .str.replace(' ', '', regex=False)
        .astype(float)
    )
    
    # Traiter date_naissance
    df['date_naissance'] = pd.NaT
    
    # Supprimer doublons métier
    df = df.drop_duplicates(subset='email', keep='first')
    
    return df

c- Fonction de generation du rapport

In [33]:
def generer_rapport(filepath):
    """
    Analyse, nettoie et affiche un rapport complet
    """
    print("="*50)
    print("        RAPPORT DE QUALITÉ DES DONNÉES")
    print("="*50)
    
    # Analyse AVANT nettoyage
    df = pd.read_csv(filepath, sep=None, engine='python')
    rapport_avant = analyser_dataset(filepath)
    
    print("\n📋 AVANT NETTOYAGE :")
    print(f"  Lignes          : {rapport_avant['nb_lignes']}")
    print(f"  Doublons email  : {rapport_avant['doublons_email']}")
    print(f"  Emails invalides: {rapport_avant['emails_invalides']}")
    print(f"  Pays non harmonisés: {rapport_avant['pays_non_harmonises']}")
    
    # Nettoyage
    df_propre = nettoyer_dataset(df.copy())
    
    # Score de qualité APRÈS nettoyage
    completude = 1 - df_propre.isnull().mean().mean()
    email_valide = df_propre['email'].str.match(r'^[^\s@]+@[^\s@]+\.[^\s@]+')
    pays_valide  = df_propre['pays'].isin(['FR', 'TN', 'MA', 'DZ'])
    validite     = (email_valide & pays_valide).mean()
    coherence    = ~((df_propre['nb_projets'] == 0) & 
                     (df_propre['chiffre_affaires_total'] > 0))
    score_global = (completude + validite + coherence.mean()) / 3
    
    print("\n📊 APRÈS NETTOYAGE :")
    print(f"  Lignes restantes : {len(df_propre)}")
    print(f"  Complétude       : {completude*100:.2f}%")
    print(f"  Validité         : {validite*100:.2f}%")
    print(f"  Cohérence        : {coherence.mean()*100:.2f}%")
    print(f"  Score global     : {score_global*100:.2f}%")
    print("="*50)
    
    return df_propre

d- Execute tout

In [34]:
# Lancer l'analyse complète
df_propre = generer_rapport('BDD tp qualité de donnée.csv')

# Sauvegarder le CSV nettoyé
df_propre.to_csv('dataset_nettoye.csv', index=False)
print("\n✅ Fichier nettoyé sauvegardé !")

        RAPPORT DE QUALITÉ DES DONNÉES

📋 AVANT NETTOYAGE :
  Lignes          : 500
  Doublons email  : 133
  Emails invalides: 130
  Pays non harmonisés: 256

📊 APRÈS NETTOYAGE :
  Lignes restantes : 367
  Complétude       : 83.13%
  Validité         : 88.28%
  Cohérence        : 94.28%
  Score global     : 88.56%

✅ Fichier nettoyé sauvegardé !


VERIFICATION DU FICHIER CSV NETTOYE

In [35]:
df_propre = pd.read_csv('dataset_nettoye.csv')

# Vérifications
print("Lignes restantes :", len(df_propre))
print("Colonnes :", list(df_propre.columns))
print()
print(df_propre.head())
print()
print(df_propre.dtypes)

Lignes restantes : 367
Colonnes : ['id', 'nom', 'prenom', 'email', 'date_naissance', 'pays', 'code_postal', 'chiffre_affaires_total', 'nb_projets', 'dernier_contact']

   id     nom  prenom                    email  date_naissance pays  \
0   1  Nguyen    Paul   paul.nguyen@domain.com             NaN   TN   
1   2  Haddad   Lucas   lucas.haddad@yahoo.com             NaN   FR   
2   3  Dupont     Awa     awa.dupont@gmail.com             NaN   TN   
3   4  Benali  Sophie  sophie.benali@gmail.com             NaN   MA   
4   5  Martin   Amine  amine.martin@domain.com             NaN   TN   

   code_postal  chiffre_affaires_total  nb_projets dernier_contact  
0        16000                  1235.0           0      2024-10-01  
1         1300                  1905.0           0             NaN  
2        75000                  1144.0          17             NaN  
3        31000                  8751.0          20             NaN  
4        75000                  4462.0          15          